<img src="images/trump-reporters.jpg" width=600>

> *President Trump said in a social media post that he is offering a $1 million reward for anyone who reports voter fraud — in past elections or in the upcoming midterm elections — that leads to a conviction.*

# 01 - First look: NYS Voting

Researchers at Stanford and Columbia have prepared for us a snapshot of the New York State voter file. The state [makes it available to anyone who requests it](https://elections.ny.gov/freedom-information-law-foil-requests) via FOIL. 

The raw data includes all 62 New York counties, including New York City. The source file is a "snapshot" from December 27, 2022. As a table, it contains 20,919,063 rows, 114 columns and 19,937,571 distinct `state_vid` values. 

The term "snapshot" here means that on December 27th of 2022, a copy of the voter file was compiled. It records people at their then-current addresses with then-current information. If after that date they move or change parties or leave the state, their data will change. So the information for December 2023 might look different for a person than it did in 2022.

IDs are never null or blank. The same ID can occur under multiple registration statuses. (Why?) The file has 981,492 more rows than distinct IDs, so a row is not necessarily a distinct voter.

We have included a "skill" for our claude agent that adapts its behavior when you ask a question about some data. Every question goes through six steps: **Question → Plan → Compute → Check → Claim → Review**. You can tell the agent to ignore this if it's too heavy-handed. 

## The data

We have created a random sample of 100,000 voters from this file. That's not 100,000 rows, but voters. The distinction is that some voters have multiple lines. For the most part, they are `PURGE`/`ACTIVE` pairs, with someone moving from one part of the state to another (the old registration remaining on the roles purged, and the new registration is acitive at their new location).

Let's load the data!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

voters = pd.read_csv("data/NY_2022_12_class_100k.csv")
type(voters)

In [ ]:
# What are the dimensions of the table?
voters.shape

In [ ]:
# Can we see the first few rows of the table?
voters.head()

In [ ]:
# Can we see the last few rows of the table?
voters.tail()

Now let's look at the `status` values. How many people are labled with each category? 

In [ ]:
voters["status"].value_counts()

## Jumping far ahead

We can take that category counting idea and look at another variable. The `reg_date` is good, although maybe too granular. Let's strip off the month and day and look at just the years...

In [ ]:
voters["reg_date"].str.split("-").str[0].head()

We can now tally the categories. See that 2016 was a good year for registrations! This list is nice but what are we irresistably drawn to do next?

In [ ]:
voters["reg_date"].str.split("-").str[0].value_counts()

That's right, a graph! We can ask our claude agent to write the code for us, putting the cell just below this one. 

A bar chart of how many voters registered in each year, in order from the earliest year to the latest.

> *Below the cell that's highlighted, create a barplot (using Altair) of the counts of people registering each year, in numerical order by year.*

In [ ]:
import altair as alt

alt.renderers.enable("png")  # draw as an image so it shows in VS Code

registrations = (
    voters["reg_date"].str.split("-").str[0].astype(int)
    .value_counts().sort_index()
    .rename_axis("year").reset_index(name="count")
)

alt.Chart(registrations).mark_bar().encode(
    x=alt.X("year:O", title="Year", sort="ascending", axis=alt.Axis(values=list(range(1925, 2030, 5)), labelAngle=0)),
    y=alt.Y("count:Q", title="Number of registrations"),
).properties(title="Voter registrations by year", width=700, height=400)

What does this plot tell you? Can we break it down a little more?

> *Make the equivalent of a "pyramid plot" where we focus on just republican vs democratic registrations and we limit the data to 1996-2022.*

In [ ]:
import altair as alt

alt.renderers.enable("png")  # draw as an image so it shows in VS Code

voters["year"] = voters["reg_date"].str.split("-").str[0].astype(int)

pyramid = (
    voters[voters["party"].isin(["DEM", "REP"]) & voters["year"].between(1996, 2022)]
    .groupby(["year", "party"]).size().reset_index(name="count")
)
# Democrats are drawn to the left of zero, Republicans to the right
pyramid["signed_count"] = pyramid["count"].where(pyramid["party"] == "REP", -pyramid["count"])

alt.Chart(pyramid).mark_bar().encode(
    y=alt.Y("year:O", title="Year of registration", sort="ascending"),
    x=alt.X("signed_count:Q", title="Number of registrations", axis=alt.Axis(labelExpr="abs(datum.value)")),
    color=alt.Color("party:N", title="Party", scale=alt.Scale(domain=["DEM", "REP"], range=["#1f77b4", "#d62728"])),
).properties(title="Democratic vs Republican registrations by year, 1996-2022", width=600, height=500)

---
## Questions from the researchers

While this is a warm up data set, there's a great deal of information to be drawn about the previous midterm election. The researchers had these questions...

> There’s so much to do with the dataset. A few ideas that come to mind:
> 
> * Compare party registration across generations. Are older voters really more conservative than younger voters?
> 
> * Look at recent registrations. Are Dems or Reps gaining ground?
> 
> * Study who are the most politically active voters. Certainly older folks, but it might also vary with political geography. (Cool GIS task here). 
> 
> * How does party registration correlate with regional demographics, economics etc?

When we're done, we can ask our friend to write the chat as a markdown file and store it in our folder. That way we have a complete transcript of what we did.

> *Can you write a markdown version of this chat in the lesson_1 folder?*